## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations. See Q37 in the <a href="https://edwarddonner.com/avatar?q=37">FAQ</a> for how to set up a separate project for your work.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...<br/>And if you post about it on LinkedIn and tag me, then I'll weigh in to amplify your achievement. If you see other students posting, please give them your encouragement too.
            </span>
        </td>
    </tr>
</table>

## Community contribution — Lab 2 (from week 1, day 3)

My contribution consist of three sets of changes. (Per Python's PEP-8 I hope my code is somewhat clear and readable enough that it does not need heavy commenting, fingers crossed).

Just for the sake of the exercise, I didn't ask any LLM to create the code or come up with the idea, so I could put just some thought to it.

### 1. To the Original lab 

- Import and use the official SDKs for Anthropic, Google, Groq, xAI, OpenRouter, and Ollama, so it can be compared side by side in the same cells.
- I added the shared `record()` helper in the Groq and Ollama cells replacing the `display` / `append` separate calls in those cells (so all the models use the record() function).

### 2. Improved workflow

This section starts after the original lab cells.. It follows the lecture prompt from Ed to try to make the workflow more sophisticated. I aimed to use only the content from the course so far and what Ed included in all the GitHub guides (e.g. Python's async), and to keep the workflow, so the essence is still there.

- I created dicts for the LLM providers/models, so parallel runs with async are easier to manage and extra per-model settings and/or more models/providers can be added later.

- Added to the judge prompt XML tags around the question, format, and responses. I read some time ago on Claude’s prompting guidance and related papers on structured prompts that this helps the LLM to understand the structure and in general is a good practice (arXiv papers 2510.22956, 2509.08182).

- Also I added a prompt dict, easier management of prompt templates, and in the case of the judge, it could be filled for the particular call with the `str.format()` method.

- New function helpers (including API-key prefix printing).

- To be faithful to the original lab, `llm_call()` keeps Ed’s `reasoning_effort="none"` behavior for `gpt-5.4-nano` , it works from a conditional.

- The async function llm_call is a starting point: it can be modified to expand/add conditions/features, for example: skip disabled models (e.g. in the LLMs dict with an "disabled" property), pull `max_tokens` / `reasoning_effort` (it could be from the prompts dict), etc, switch SDKs (I coded this new section to use the OpenAISDK. As recreating an improved version of the original lab code.)

### 3. Additional agentic design pattern

- I added an additional agentic workflow design pattern at the end, as an example to start a version 3 of the code of this lab (v1 the original, v2 the improved workflow, v3 adding additional patterns or calls asking new things). The added pattern is **routing**. 

It works this way:

1. Models vote using the agentic workflow parallel design pattern on which model should write the opening question (an LLM council).
2. Votes are counted. On a tie, `max()` keeps the first model that reached that count. (This could be changed but for the moment let's say it's "destiny").
3. That code routes the question-generation prompt to the winner model.

In [ ]:
# Start with imports - ask the Cursor Agent to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from google import genai
from groq import Groq
from xai_sdk import Client as grok_client
from xai_sdk.chat import user as grok_user
from openrouter import OpenRouter
from ollama import Client as ollama_chat
from IPython.display import Markdown, display

In [ ]:
# Always remember to do this!
load_dotenv(override=True)

In [ ]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}") 
else:
    print("OpenRouter API Key not set (and this is optional)")


In [ ]:
request = """
Please come up with a challenging, nuanced question with a succinct answer,
that I can ask a number of LLMs to evaluate their intelligence.
Not a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.
Include in your question that the answer must be short.
"""
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [ ]:
messages

In [ ]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
question = response.choices[0].message.content
display(Markdown(question))

## Calling LLMs from multiple providers

We are about to call LLMs from many other providers.
They all provide API endpoints that are compatible with OpenAI, as explained in Guide 9 in the guides folder.
So we can simply use these endpoints as if we are using OpenAI.

Please note:

I'm going to use lots of LLMs from different providers, but you don't need to! This is only to show their abilities.

In [ ]:
# OpenAI Compatible URLs

ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROK_BASE_URL = "https://api.x.ai/v1"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [ ]:
# OpenAI client libraries with the right base_url and key
# If this surprises you, please see Guide 9 in the Guides folder!

anthropic = OpenAI(api_key=anthropic_api_key, base_url=ANTHROPIC_BASE_URL)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=DEEPSEEK_BASE_URL)
gemini = OpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL)
groq = OpenAI(api_key=groq_api_key, base_url=GROQ_BASE_URL)
grok = OpenAI(api_key=grok_api_key, base_url=GROK_BASE_URL)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_BASE_URL)
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [ ]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

In [ ]:
def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(answer))

In [ ]:
# The API we know well
# Reasoning effort can be none, low, medium, high, or xhigh

model_name = "gpt-5.4-nano"

response = openai.chat.completions.create(model=model_name, messages=messages, reasoning_effort="none")
answer = response.choices[0].message.content

record(model_name, answer)

In [ ]:
# Anthropic

model_name = "claude-sonnet-4-6"

# via OpenAI SDK compatibility
# response = anthropic.chat.completions.create(model=model_name, messages=messages)
# answer = response.choices[0].message.content

# via Anthropic SDK
anthropic_api_call = Anthropic()
response = anthropic_api_call.messages.create(model=model_name, messages=messages, max_tokens=1024)
answer = response.content[0].text

record(model_name, answer)

In [ ]:
# Gemini

model_name = "gemini-3.1-flash-lite"

# via OpenAI SDK compatibility
# response = gemini.chat.completions.create(model=model_name, messages=messages)
# answer = response.choices[0].message.content

# via Google SDK
gemini_sdk_call = genai.Client()
response = gemini_sdk_call.interactions.create(model=model_name, input=question)
answer = response.output_text

record(model_name, answer)

In [ ]:
# DeepSeek
# DeepSeek has no official Python SDK, so using the OpenAI compatible client.

model_name = "deepseek-v4-flash"

response = deepseek.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

In [ ]:
# Groq

model_name = "openai/gpt-oss-120b"

# via OpenAI SDK compatibility
# response = groq.chat.completions.create(model=model_name, messages=messages)
# answer = response.choices[0].message.content

# via Groq SDK
groq_sdk_call = Groq()
response = groq_sdk_call.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(f"Groq - {model_name}", answer)

In [ ]:
# OpenRouter

model_name = "moonshotai/kimi-k2.6"

# via OpenAI SDK compatibility
# response = openrouter.chat.completions.create(model=model_name, messages=messages)
# answer = response.choices[0].message.content

# via OpenRouter SDK
openrouter_sdk_call = OpenRouter(api_key=openrouter_api_key)
response = openrouter_sdk_call.chat.send(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(f"OpenRouter - {model_name}", answer)


## For the next cell, we will use Ollama

Ollama runs a local web service that gives an OpenAI compatible endpoint,  
and runs models locally using high performance C++ code.

If you don't have Ollama, install it here by visiting https://ollama.com then pressing Download and following the instructions.

After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"

You might need to restart Cursor (and maybe reboot). Then open a Terminal (control+\`) and run `ollama serve`

Useful Ollama commands (run these in the terminal, or with an exclamation mark in this notebook):

`ollama pull <model_name>` downloads a model locally  
`ollama ls` lists all the models you've downloaded  
`ollama rm <model_name>` deletes the specified model from your downloads

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Super important - ignore me at your peril!</h2>
            <span style="color:#ff7800;">Many models on Ollama are FAR too large for your home computer. Be sure to browse the models on the Ollama website. Look to use models that are size 3GB or less unless you know better; llama3.2 is a great first choice. Don't pick models that end in :cloud; that's something different (a cloud inference service, like Groq).
            </span>
        </td>
    </tr>
</table>

In [ ]:
!ollama pull llama3.2

In [ ]:
import requests
requests.get('http://localhost:11434').content

In [ ]:
import requests
models = requests.get('http://localhost:11434/v1/models').json()
for model in models.get("data"):
    print(model.get("id"))

In [ ]:
# Local Ollama - Llama3.2:1b

model_name = "llama3.2:1b"

# via OpenAI SDK compatibility
# response = ollama.chat.completions.create(model=model_name, messages=messages)
# answer = response.choices[0].message.content

# via Ollama SDK
ollama_sdk_call = ollama_chat()
response = ollama_sdk_call.chat(model=model_name, messages=messages)
answer = response.message.content

record(f"Local Ollama: {model_name}", answer)

In [ ]:
# Local Ollama - gpt-oss:latest

model_name = "gpt-oss:latest"

# via OpenAI SDK compatibility
# response = ollama.chat.completions.create(model=model_name, messages=messages)
# answer = response.choices[0].message.content

# via Ollama SDK
response = ollama_sdk_call.chat(model=model_name, messages=messages)
answer = response.message.content

record(f"Local Ollama: {model_name}", answer)

In [ ]:
# Local Ollama - gemma4:latest

model_name = "gemma4:latest"

# via OpenAI SDK compatibility
# response = ollama.chat.completions.create(model=model_name, messages=messages)
# answer = response.choices[0].message.content

# via Ollama SDK
response = ollama_sdk_call.chat(model=model_name, messages=messages)
answer = response.message.content

record(f"Local Ollama: {model_name}", answer)

In [ ]:
# So where are we?

print(len(competitors))
print(competitors)
print(answers)


In [ ]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


In [ ]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
print(together)

In [ ]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

<question>
{question}
</question>

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
<format>
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}
</format>

Here are the responses from each competitor:

<responses>
{together}
</responses>

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [ ]:
print(judge)

In [ ]:
judge_messages = [{"role": "user", "content": judge}]

## And now for Grok!

Branded as "The most truth-seeking large language model in the world".. so let's use it as our LLM as a judge

In [ ]:
# Judgement time!
# Grok is "The most truth-seeking large language model in the world."

model_name = "grok-4.3"

# via OpenAI SDK compatibility
# response = grok.chat.completions.create(model=model_name, messages=judge_messages)
# results = response.choices[0].message.content
# print(results)

# via Grok SDK
grok_sdk_call = grok_client()
response = grok_sdk_call.chat.create(model=model_name).append(grok_user(judge))
results = response.sample().content
print(results)


In [ ]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

Answer: Patterns this used: a combination of **prompt chaining**  and **parallelization**

In [ ]:
import os
import json
import asyncio

from dotenv import load_dotenv
from openai import AsyncOpenAI
from IPython.display import Markdown, display


load_dotenv(override=True)


llms_list = {
    "OpenAI": {
        "models": ["gpt-5.4-nano"],
        "api_base_url": "",
        "api_key": os.getenv('OPENAI_API_KEY'),
        "safe_key_reveal": 8
    },
    "Anthropic": {
        "models": ["claude-sonnet-4-6"],
        "api_base_url": "https://api.anthropic.com/v1/",
        "api_key": os.getenv('ANTHROPIC_API_KEY'),
        "safe_key_reveal": 7
    },
    "Google": {
        "models": ["gemini-3.1-flash-lite"],
        "api_base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
        "api_key": os.getenv('GOOGLE_API_KEY'),
        "safe_key_reveal": 2
    },
    "DeepSeek": {
        "models": ["deepseek-v4-flash"],
        "api_base_url": "https://api.deepseek.com/v1",
        "api_key": os.getenv('DEEPSEEK_API_KEY'),
        "safe_key_reveal": 3
    },
    "Groq": {
        "models": ["openai/gpt-oss-120b"],
        "api_base_url": "https://api.groq.com/openai/v1",
        "api_key": os.getenv('GROQ_API_KEY'),
        "safe_key_reveal": 4
    },
    "OpenRouter": {
        "models": ["moonshotai/kimi-k2.6"],
        "api_base_url": "https://openrouter.ai/api/v1",
        "api_key": os.getenv('OPENROUTER_API_KEY'),
        "safe_key_reveal": 6
    },
    "Local Ollama": {
        "models": ["llama3.2:1b", "gemma4:latest", "gpt-oss:latest"],
        "api_base_url": "http://localhost:11434/v1",
        "api_key": "ollama",
        "safe_key_reveal": 2
    },
    "Grok": {
        "models": ["grok-4.3"],
        "api_base_url": "https://api.x.ai/v1",
        "api_key": os.getenv('GROK_API_KEY'),
        "safe_key_reveal": 4
    },
}


# Print the key prefixes to help with any debugging
for llm_provider, provider_details in llms_list.items():
    print(f"{llm_provider} API Key", end=" ")
    if provider_details.get("api_key"):
        print(f"exists and begins {provider_details.get("api_key")[:provider_details.get("safe_key_reveal")]}")
    else:
        print(f"not set{" (and this is optional)" if llm_provider != "OpenAI" else ""}")
        

lab2_default_prompts = {
    "initial_question": """
                  Please come up with a challenging, nuanced question with a succinct answer,
that I can ask a number of LLMs to evaluate their intelligence.
Not a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.
Include in your question that the answer must be short.
                 Answer only with the question, no explanation.
                        """,
    "judge": """
                You are judging a competition between {len_competitors} competitors.
Each model has been given this question:

<question>
{question}
</question>

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
<format>
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}
</format>

Here are the responses from each competitor:

<responses>
{together}
</responses>

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks.
            """
}

In [ ]:
competitors = []
answers = []


async def record(provider_n_model_name: str, answer: str, print_answer: bool):
    competitors.append(provider_n_model_name)
    answers.append(answer)
    if print_answer:
        display(Markdown(f"{provider_n_model_name} answer: {answer}"))


async def llm_call(models_to_call: str | tuple, prompt: str):
    single_model = True if isinstance(models_to_call, tuple) else False

    provider = models_to_call[0] if single_model else models_to_call
    models_list = [models_to_call[1]] if single_model else llms_list.get(provider).get("models")

    openai_sdk_async_client = AsyncOpenAI() if provider == "OpenAI" else AsyncOpenAI(api_key=llms_list.get(provider).get("api_key"), base_url=llms_list.get(provider).get("api_base_url"))
    
    for model in models_list:
        # Random colour for the provider/model name so parallel async logs are easier to scan. The color index is chosen to stay readable on dark and light backgrounds
        prov_mod_colour = f"\033[38;5;{82+sum(map(ord, model))%150}m{provider} - {model}\033[0m"
        try:
            print("\x1B[3m" + "Sending request/prompt to: "+ "\x1B[0m" + f"{prov_mod_colour}")
            response = await openai_sdk_async_client.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}], **({"reasoning_effort": "none"} if model == "gpt-5.4-nano" else {}))
            answer = response.choices[0].message.content
        except Exception as e:
            print(f"Error from {prov_mod_colour}: {e}")
        else:
            print(f"{prov_mod_colour} " + "\033[4m" + "response received." + "\033[0m")
            if single_model:
                return answer
            else:
                await record(f"{provider}: {model}", answer, False)

In [ ]:
question = await llm_call(("OpenAI", "gpt-5.4-mini"), lab2_default_prompts.get("initial_question"))
display(Markdown(question))

In [ ]:
# Parallelization, as in the previous section, but with async
llms_in_parallel = [llm_call(provider, question) for provider in llms_list.keys() if provider != "Grok"]
results = await asyncio.gather(*llms_in_parallel)

In [ ]:
# So where are we?

print(len(competitors))
print(competitors)
print(answers)

In [ ]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")

In [ ]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

print(together)

In [ ]:
# Judgement time!
# Grok is "The most truth-seeking large language model in the world."

results = await llm_call(("Grok", "grok-4.3"), lab2_default_prompts.get("judge").format(len_competitors=len(competitors), question=question, together=together))
print(results)

In [ ]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

### Now adding another Agentic Workflow Design Pattern

#### Routing:

In [ ]:
# I know there is some inconsistency in the type hits in my code here, as I added some type hints to some variables/functions and not to others like outputs. For this exercise I mostly annotated function parameters where I think it helps with clarification

list_of_models = [model for provider in llms_list.values() for model in provider.get("models", [])]

# I added the "```json" instruction because it happened a few times when I was testing the approach, especially from the local models
llm_council_prompt = f"""
As of today, 31 August 2026, choose which model from this list is best suited to write "a challenging, nuanced question with a short answer, which will then be used to compare several LLMs. The question should be thought-provoking, not a math puzzle."

<list_of_models>{", ".join(list_of_models)}</list_of_models>

Reply with JSON only, in exactly this shape:
{{"model_selected": "the model you selected", "reason": "one sentence"}}

Don't answer the question itself. Exclude from the output any chain-of-thought information or steps you did. No markdown, no code fences, no extra braces. Don't prefix the output with the following characters: ```json
"""

previous_competitors = competitors.copy()
previous_answers = answers.copy()

competitors = []
answers = []


llms_in_parallel = [llm_call(provider, llm_council_prompt) for provider in llms_list.keys()]
await asyncio.gather(*llms_in_parallel)

In [ ]:
for competitor, answer in zip(competitors, answers):
    print(f"------\n-Competitor: {competitor}\n-Answer: {answer}")

In [ ]:
vote_results = []
winner = ""

# I thought of using json.JSONDecoder().raw_decode() because in a few occasions the Local LLM returned an extra character, like }}.
# But then I considered there are a few other verifications that could be done, so I decided to go for a try/except
for index, llm_voter_answer in enumerate(answers):
    try:
        vote = json.loads(llm_voter_answer).get("model_selected")
    except Exception as e:
        print(f"|----\n-Could not parse the vote from - {competitors[index]}\n-Answer: {answers[index]}\n-Error: {e} \n")
    else:
        vote_results.append(vote)


winner = max(set(vote_results), key=vote_results.count)
print(f"From the LLM Council, the model selected for the next task is: {winner}")

In [ ]:
def get_provider(model_name: str):
    return next((provider for provider, provider_details in llms_list.items() if model_name in provider_details.get("models", [])), None)


question = await llm_call((get_provider(winner), winner), lab2_default_prompts.get("initial_question"))
display(Markdown(question))

- Other ideas I considered for adding another agentic workflow design pattern:
    - Same routing idea, but vote on which model should be the judge, and why.
    - "Prompt chaining" so a random model is selected to write the initial question and another random model is selected as the judge prompt. Then adding the "Evaluator-Optimizer" so the others LLMs act as evaluators and accept or reject the proposed question.
    - Ask the models, using Anthropic’s pattern definitions, which extra agentic workflow design pattern they would add and why, then implement that choice with "Orchestrator–Workers" design pattern.
    - Ask the models to vote on call order by latency (fastest API first). Even if it only saves milliseconds, I think it is an interesting experiment.
    - Just for fun and comparison, for one or two models, call the HTTP API directly with the `requests` library (`POST`, `GET`, and so on).

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>